# Chapter 3 — Is an overdensity made of stars?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/03_star_galaxy_populations.ipynb)

**Goal.** Chapter 1 showed that faint "stars" are partly galaxies. Galaxies cluster, so a group or cluster of
galaxies produces a compact overdensity of "stars" that looks exactly like a faint satellite on a sky map. Here
we compare two overdensities: the galaxy cluster **Abell 370** (redshift z = 0.375) and the dwarf galaxy
**Boötes I**. Both are highly significant excesses of SDSS point sources. Our question is about the *excess
population* — not about any single object.

**You will learn**
1. to test an overdensity in surveys of different depth and image quality;
2. to describe the excess in colour–colour space as a mixture of a stellar template and a galaxy template, and
   measure its **star fraction** $f_\star$ with a likelihood;
3. how photometric errors, and their correlations between colours, enter such templates;
4. to compare the shape (PSF − model magnitude) of the excess objects with that of real stars of the same magnitude.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats                               # Poisson tail probabilities and chi^2 quantiles
from scipy.optimize import minimize                   # maximum-likelihood fit of the two-template model
from scipy.ndimage import gaussian_filter             # smoothing of the galaxy template
from scipy.spatial import cKDTree                     # nearest-neighbour crossmatch to HST

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


def lsmag(flux):
    """Legacy Surveys fluxes are in nanomaggies: m = 22.5 - 2.5 log10(flux)."""
    return 22.5 - 2.5 * np.log10(np.where(flux > 0, flux, np.nan))   # non-positive fluxes -> NaN magnitude


A370 = (39.970, -1.577)                               # (ra, dec) [deg]: centre of the a370_* extractions (0.33° radius)

## 3.1 The same field in four surveys

For each survey we keep what *that survey* calls a star (the classifiers of Chapter 1) with 14 < r < 23, and
measure the surface density in rings around the cluster centre, relative to the density at 8′–20′.

In [ ]:
# Each survey's catalogue, r magnitude and own star/galaxy decision, as in Chapter 1 (observed magnitudes)
surveys = {}
d = load('a370_sdss'); surveys['SDSS'] = (d, d.psfmag_r.to_numpy(), (d.type == 6).to_numpy())
d = load('a370_ps1'); surveys['PS1'] = (d, d.rpsfmag.to_numpy(), (d.rpsfmag - d.rkronmag < 0.05).to_numpy())
d = load('a370_ls'); surveys['LS'] = (d, lsmag(d.flux_r.to_numpy()), (d.type == 'PSF').to_numpy())
d = load('a370_hsc'); surveys['HSC'] = (d, d.r_psfflux_mag.to_numpy(), (d.i_extendedness_value < 0.5).to_numpy())

# Rings around the cluster centre; the last one, 8′-19.8′, reaches the edge of the 0.33° = 19.8′ extraction and
# serves as the background.
edges = np.array([0, 1, 2, 4, 8, 19.8])                          # arcmin
area = np.pi * np.diff(edges**2)                                 # ring areas [arcmin^2] (flat sky)
counts, inner = {}, {}
fig, ax = plt.subplots(figsize=(7, 4.5))
for k, (d, m, cl) in surveys.items():
    R = np.hypot(*tangent(d.ra.to_numpy(), d.dec.to_numpy(), *A370)) * 60   # arcmin from the cluster centre
    s = cl & (m > 14) & (m < 23)                                 # "stars" of this survey, 14 < r < 23
    n = np.histogram(R[s], edges)[0]                             # "stars" per ring
    bg = n[-1] / area[-1]                                        # stars per arcmin^2 at 8'-20'
    counts[k] = n
    inner[k] = (n[0] + n[1], bg * np.pi * 2**2)                  # observed and expected inside 2'
    # density relative to the background, with Poisson errors sqrt(n) (sqrt(1) for an empty ring)
    ax.errorbar(0.5 * (edges[1:] + edges[:-1]), n / area / bg, np.sqrt(np.maximum(n, 1)) / area / bg, marker='o',
                label=f'{k}: {bg * 3600:,.0f} "stars" per deg²')
ax.axhline(1, c='k', lw=0.6)
ax.set(xscale='log', xlabel="distance from Abell 370 [arcmin]", ylabel='density / background', title='point sources, 14 < r < 23')
ax.legend(fontsize=8)
plt.tight_layout()

for k, (n, b) in inner.items():
    p = stats.poisson.sf(n - 1, b)                               # P(N >= n | background b)
    up = stats.chi2.ppf(0.95, 2 * (n + 1)) / 2                   # 95% upper limit on the Poisson mean, given n
    print(f'{k:5s}: {n:3d} inside 2′, background {b:5.1f}; P(N >= n) = {p:.1e}; 95% upper limit on ratio {up / b:.2f}')
# Tests (STRUCTURE.md), with measured values:
# the SDSS "star" excess inside 2′ is significant, p < 1e-6 (71 vs 21.6 expected: p = 3.6e-17);
assert stats.poisson.sf(inner['SDSS'][0] - 1, inner['SDSS'][1]) < 1e-6
# HSC sees no excess: the 95% upper limit on its density ratio inside 2′ is below 2 (5 vs 5.9 expected: 1.77).
n, b = inner['HSC']
assert stats.chi2.ppf(0.95, 2 * (n + 1)) / 2 / b < 2

**Figure 3.1.** Radial profiles of "stars" around Abell 370 in four surveys (`a370_{sdss,ps1,ls,hsc}`): objects that
each survey calls a star (its classifier from Chapter 1) with 14 < r < 23 in its own observed r magnitude. Surface
density in rings 0–1′, 1–2′, 2–4′, 4–8′ and 8′–19.8′, divided by the density in the outer ring (which is 1 by
construction; horizontal line), against ring centre (log axis); error bars are Poisson, $\sqrt{N}$. Blue SDSS, orange
PS1, green LS, red HSC; the legend gives each background density, from 6180 (SDSS), 8377 (PS1) and 5834 (LS) down to
1698 (HSC) "stars" per deg². SDSS and PS1 see a strong central excess: 71 and 86 objects inside 2′ against 21.6 and
29.2 expected, P(N ≥ n) = 3.6 × 10⁻¹⁷ and 1.5 × 10⁻¹⁷ — by the Poisson probability alone a "detection" far beyond
5σ. LS, built from deeper DECam images with a better model fit, shows a weak, insignificant excess (26 against 20.4,
P = 0.13); HSC (0.6″ seeing, 8-m telescope) shows none (5 against 5.9; 95% upper limit on the density ratio 1.77).
Note also the background densities: at the same magnitude limit the surveys disagree by a factor of several on how
many "stars" there are, because their contamination and completeness differ (Chapter 1).

The lesson is not "use the deeper survey" — often there is none. It is that we need tests that can be applied
to the excess *in the survey we have*.

## 3.2 The colours of the excess: a two-template likelihood

Bin the objects of a region in colour–colour space $(g-r, r-i)_0$. In the on-region (the overdensity), the expected
counts in bin $j$ are

$$ \lambda_j = B_j + n_\star T^\star_j + n_G T^G_j, $$

where $B_j = \alpha N^{\rm off}_j$ is the background predicted from the off-region (scaled by the area ratio α),
$T^\star$ and $T^G$ are normalised templates for stars and galaxies, and $n_\star, n_G \ge 0$ are the numbers of
excess stars and galaxies. The Poisson log-likelihood of the observed counts $N_j$ is
$\ln\mathcal L = \sum_j (N_j \ln\lambda_j - \lambda_j)$. The star fraction is $f_\star = n_\star/(n_\star + n_G)$; we
find its 68% interval from the profile likelihood ($2\Delta\ln\mathcal L < 1$, maximising over $n_\star + n_G$).

**Templates, from COSMOS** (Chapter 1), in SDSS:
- galaxy template: SDSS "stars" with 20 < r < 22.5 that HST shows to be galaxies — exactly the contaminants;
- stellar template: bright HST-confirmed stars (16 < r < 20, small errors), each given random noise drawn from
  the errors of the *target* stars. Noise is added to g, r and i separately, so the colours inherit the right
  correlation: an error in r moves g − r and r − i in opposite directions, ${\rm Cov}(g-r, r-i) = -\sigma_r^2$.

In [ ]:
rng = np.random.default_rng(3)                                  # fixed seed: the noisy star templates are reproducible
hst = load('cosmos_hst')
hst = hst[hst.mu_class.isin([1, 2])]                             # HST galaxies (1) and stars (2); artifacts dropped
cs = load('cosmos_sdss')                                         # SDSS DR9, all types, r < 24, 0.6° around COSMOS
# Flat-sky crossmatch within 1″: RA scaled by cos(2.2°), the declination of COSMOS
c = np.cos(np.radians(2.2))
dist, idx = cKDTree(np.c_[hst.ra * c, hst.dec]).query(np.c_[cs.ra * c, cs.dec], distance_upper_bound=1 / 3600)
matched = np.isfinite(dist)                                      # unmatched sources have dist = inf
hst_star = np.zeros(len(cs), bool)
hst_star[matched] = hst.mu_class.to_numpy()[idx[matched]] == 2


def dered(d):
    """Dereddened g, r, i and their errors.

    Returns ([g0, r0, i0], [sigma_g, sigma_r, sigma_i]) as numpy arrays [mag].
    """
    return [(d[f'psfmag_{b}'] - d[f'extinction_{b}']).to_numpy() for b in 'gri'], [d[f'psfmagerr_{b}'].to_numpy() for b in 'gri']


# Colour-colour bins of 0.2 mag: (g-r)_0 from -0.5 to 2.5 (15 bins), (r-i)_0 from -0.5 to 1.9 (12 bins)
gb, ib = np.arange(-0.5, 2.51, 0.2), np.arange(-0.5, 2.01, 0.2)
hist = lambda gr, ri: np.histogram2d(gr, ri, [gb, ib])[0]        # counts, shape (15, 12)
(Gc, Rc, Ic), _ = dered(cs)
point = (cs.type == 6).to_numpy() & matched                      # SDSS "stars" with an HST match
faint = (Rc > 20) & (Rc < 22.5)                                  # the magnitude range of the targets below
# Galaxy template: SDSS "stars" that HST shows to be galaxies, 20 < r_0 < 22.5 (961 objects). Smoothed by a Gaussian
# of 0.7 bin (0.14 mag), plus a floor of 1e-3 per bin so that no colour has zero galaxy probability; normalised to
# unit sum (probability per bin).
Tg = gaussian_filter(hist((Gc - Rc)[point & ~hst_star & faint], (Rc - Ic)[point & ~hst_star & faint]), 0.7) + 1e-3
Tg /= Tg.sum()
# Stellar template source: HST stars that SDSS calls stars, 16 < r_0 < 20 (2087 stars, median errors ~0.02 mag)
bright = point & hst_star & (Rc > 16) & (Rc < 20)


def star_template(errs, n_copies=20):
    """Bright HST stars with noise drawn from the target's errors (errs = [sigma_g, sigma_r, sigma_i] arrays).

    Each bright star is copied n_copies times; each copy takes the (sigma_g, sigma_r, sigma_i) of one randomly chosen
    target star and gets independent Gaussian noise in g, r and i. Because both colours share the r error, the
    template has Cov(g-r, r-i) = -sigma_r^2. Returns a normalised (15, 12) histogram with a 1e-3 floor per bin.
    """
    k = rng.integers(0, len(errs[0]), size=(n_copies, bright.sum()))   # index of a target star, per copy
    g, r, i = [m[bright] + rng.normal(size=k.shape) * e[k] for m, e in zip((Gc, Rc, Ic), errs)]
    T = hist((g - r).ravel(), (r - i).ravel()) + 1e-3
    return T / T.sum()


def fit_fstar(on, B, Ts):
    """Best (n_star, n_gal) and the profile likelihood of f_star = n_star / (n_star + n_gal).

    on: observed on-region counts; B: expected background counts (alpha * off); Ts: star template; all (15, 12).
    Returns the best (n_star, n_gal), the f_star grid, 2 Delta ln L on it and the 68% interval (2 Delta ln L < 1).
    """
    # Poisson log-likelihood sum(N ln lambda - lambda), dropping the constant ln N!; 1e-12 guards against ln(0)
    lnl = lambda ns, ng: np.sum(on * np.log(B + ns * Ts + ng * Tg + 1e-12) - (B + ns * Ts + ng * Tg))
    best = minimize(lambda n: -lnl(*n), [10, 10], method='L-BFGS-B', bounds=[(0, None)] * 2).x   # n_star, n_gal >= 0
    # profile: on a grid of f_star in steps of 0.01, maximise over the total excess t = n_star + n_gal >= 0
    fs = np.linspace(0, 1, 101)
    prof = np.array([-minimize(lambda t: -lnl(t[0] * f, t[0] * (1 - f)), [best.sum()], method='L-BFGS-B',
                               bounds=[(0, None)]).fun for f in fs])
    prof = 2 * (prof.max() - prof)
    inside = fs[prof < 1]
    return best, fs, prof, (inside.min(), inside.max())


def region(name, centre, r_on, r_off, point_only):
    """Binned colours of an on-region (R < r_on) and the area-scaled background from an annulus r_off [arcmin].

    Objects with 20 < r_0 < 22.5 only; point_only keeps SDSS type 6 (for files that also hold galaxies). Returns
    N_on, B, the star template blurred with the on-region errors, and the on/off colours for plotting.
    """
    d = load(name)
    if point_only:
        d = d[d.type == 6]
    (g, r, i), e = dered(d)
    R = np.hypot(*tangent(d.ra.to_numpy(), d.dec.to_numpy(), *centre)) * 60
    s = (r > 20) & (r < 22.5)                                    # same magnitude range as the templates
    on, off = s & (R < r_on), s & (R > r_off[0]) & (R < r_off[1])
    alpha = r_on**2 / (r_off[1]**2 - r_off[0]**2)               # area ratio A_on / A_off
    N_on, B = hist((g - r)[on], (r - i)[on]), alpha * hist((g - r)[off], (r - i)[off])
    Ts = star_template([x[on] for x in e])
    return N_on, B, Ts, (g - r)[on], (r - i)[on], (g - r)[off], (r - i)[off]


lvdb = load('lvdb').set_index('key')
boo = lvdb.loc['bootes_1']
# Abell 370: SDSS point sources within 2′, background 8′-19.8′. Boötes I: within r_h (9.97′), background 60′-120′
# (bootes1_sdss holds point sources only).
cases = {'Abell 370 (R < 2′)': region('a370_sdss', A370, 2, (8, 19.8), True),
         f'Boötes I (R < r_h)': region('bootes1_sdss', (boo.ra, boo.dec), boo.rhalf, (60, 120), False)}
results = {}
for k, (N_on, B, Ts, *_) in cases.items():
    best, fs, prof, ci = fit_fstar(N_on, B, Ts)
    results[k] = (best, fs, prof, ci)
    print(f'{k:20s}: on {N_on.sum():4.0f}, background {B.sum():6.1f}, excess {N_on.sum() - B.sum():6.1f}; '
          f'fit n_star = {best[0]:6.1f}, n_gal = {best[1]:5.1f}; f_star 68% interval [{ci[0]:.2f}, {ci[1]:.2f}]')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ext = [gb[0], gb[-1], ib[0], ib[-1]]
# Left: template contours at 0.005, 0.02 and 0.05 of the objects per 0.2 x 0.2-mag bin. Ts is the last template
# built in the loop above, i.e. the one blurred with the Boötes I on-region errors.
ax[0].contour(0.5 * (gb[1:] + gb[:-1]), 0.5 * (ib[1:] + ib[:-1]), Ts.T, levels=np.array([0.005, 0.02, 0.05]), colors='C0')
ax[0].contour(0.5 * (gb[1:] + gb[:-1]), 0.5 * (ib[1:] + ib[:-1]), Tg.T, levels=np.array([0.005, 0.02, 0.05]), colors='C1')
# the on-region objects: Abell 370 as black squares, Boötes I as green circles
for (k, v), mk, cc in zip(cases.items(), ['s', 'o'], ['k', 'C2']):
    ax[0].scatter(v[3], v[4], s=10, marker=mk, c=cc, label=f'{k}: on-region')
ax[0].set(xlim=(-0.5, 2.3), ylim=(-0.5, 1.8), xlabel='$(g-r)_0$', ylabel='$(r-i)_0$',
          title='templates: stars (blue), galaxies (orange)')
ax[0].legend(fontsize=7)
# middle and right: on - off counts per bin, on a diverging scale fixed at +-15
for a, (k, (N_on, B, *_)) in zip(ax[1:], cases.items()):
    im = a.imshow((N_on - B).T, origin='lower', extent=ext, cmap='RdBu_r', vmin=-15, vmax=15, aspect='auto')
    a.set(xlabel='$(g-r)_0$', ylabel='$(r-i)_0$', title=f'on − off: {k}', xlim=(-0.5, 2.3), ylim=(-0.5, 1.8))
    plt.colorbar(im, ax=a)
plt.tight_layout()

# Second figure: the profile likelihood of f_star for both regions; the dotted line marks the 68% level
fig, ax = plt.subplots(figsize=(6, 4))
for k, (best, fs, prof, ci) in results.items():
    ax.plot(fs, prof, label=k)
ax.axhline(1, c='k', ls=':', lw=0.8)
ax.set(xlabel='star fraction of the excess $f_\\star$', ylabel='$2\\Delta\\ln\\mathcal{L}$', ylim=(0, 20))
ax.legend()
plt.tight_layout()

# Test (STRUCTURE.md): f_star(Boötes I) > 0.7, f_star(A370 SDSS) < 0.3, and the 68% intervals are disjoint.
# Measured: Boötes I [0.97, 1.00], Abell 370 [0.00, 0.28].
(ca, cb_) = results['Abell 370 (R < 2′)'][3], results['Boötes I (R < r_h)'][3]
assert cb_[0] > 0.7 and ca[1] < 0.3 and ca[1] < cb_[0]

**Figure 3.2.** Colour–colour diagrams, $(r-i)_0$ against $(g-r)_0$, of SDSS point sources with 20 < $r_0$ < 22.5.
Left: the two templates, as contours at 0.005, 0.02 and 0.05 of the objects per 0.2 × 0.2-mag bin — blue: bright
(16 < r < 20) HST-confirmed COSMOS stars blurred by the errors of faint targets (the template drawn uses the Boötes I
errors); orange: COSMOS objects that SDSS calls stars and HST shows to be galaxies — and the objects in the two
on-regions: Abell 370 within 2′ (black squares, 48 objects) and Boötes I within $r_h$ (green circles, 430). Middle
and right: on − off counts in 0.2-mag bins (off-region scaled by area: 8′–19.8′ for Abell 370, 60′–120′ for Boötes I;
colour scale ±15 objects per bin), with excesses of 36 and 224 objects. The Abell 370 excess lies at g − r ≈ 1.2–2.0
and r − i ≈ 0.3–0.7, *below* the M-dwarf branch of the stellar locus: the red sequence of the cluster's early-type
galaxies. The Boötes I excess lies on the blue part of the stellar locus, $(g-r)_0 \lesssim 0.6$, where old,
metal-poor stars are.

**Figure 3.3.** Profile likelihood $2\Delta\ln\mathcal L$ of the star fraction $f_\star$ of the excess (two-template
Poisson fit of the binned on-region colours, maximised over $n_\star + n_G$ at each $f_\star$ on a 0.01 grid), for
Abell 370 within 2′ (blue) and Boötes I within $r_h$ (orange); the dotted line marks $2\Delta\ln\mathcal L = 1$. The
68% intervals are [0.00, 0.28] for Abell 370 (best fit $n_\star$ = 0.1, $n_G$ = 37.8) and [0.97, 1.00] for Boötes I
($n_\star$ = 166.4, $n_G$ = 0.0). The two intervals do not overlap: the Abell 370 excess is consistent with galaxies
only, the Boötes I excess with stars only.

Not every Boötes I star is absorbed by the fit: ~25% of its excess sits at $(g-r)_0 < 0$ (blue
horizontal-branch stars and blue stragglers) or at the extreme of the error distribution, where neither
template has support. A complete model would add a template for these populations; for the question asked here —
stars or galaxies? — it does not matter.

Two cautions. The galaxy template describes field galaxies that leak into the SDSS star sample at 20 < r < 22.5;
a cluster at another redshift has different colours, and at some redshifts its red sequence crosses the stellar
locus — colours alone then fail. And this colour test does not use the CMD, which is the strongest evidence
that an excess is a stellar population at one distance (Chapter 2; Chapter 6).

## 3.3 The shape of the excess

The excess objects can also be compared with real stars in shape. For all SDSS objects (any type) with
20 < r < 22.5, take $\Delta m = r_{\rm PSF} - r_{\rm cModel}$, subtract the off-region distribution from the
on-region one, and compare with HST-confirmed stars and galaxies *of the same magnitude* from COSMOS (bright stars
would be a wrong reference: their Δm distribution is much narrower).

In [ ]:
a = load('a370_sdss')                                           # all SDSS objects (types 3 and 6) around Abell 370
R = np.hypot(*tangent(a.ra.to_numpy(), a.dec.to_numpy(), *A370)) * 60
r = (a.psfmag_r - a.extinction_r).to_numpy()
dm = (a.psfmag_r - a.cmodelmag_r).to_numpy()                    # Delta m [mag]; extinction cancels in the difference
s = (r > 20) & (r < 22.5)
db = np.arange(-0.4, 1.21, 0.1)                                  # 0.1-mag bins of Delta m (values outside not counted)
# excess = on (R < 2′) minus off (8′-19.8′) scaled by the area ratio 2^2 / (19.8^2 - 8^2)
excess = np.histogram(dm[s & (R < 2)], db)[0] - 4 / (19.8**2 - 64) * np.histogram(dm[s & (R > 8) & (R < 19.8)], db)[0]
# references: COSMOS SDSS objects of any type in the same magnitude range, split by HST class
dmc = (cs.psfmag_r - cs.cmodelmag_r).to_numpy()
fc = matched & faint
fig, ax = plt.subplots(figsize=(7, 4.3))
frac = lambda h: h / h.sum()                                    # fraction of objects per 0.1-mag bin
# step(where='post') draws each value from its bin's left edge db[:-1]
ax.step(db[:-1], frac(excess), where='post', lw=2.5, c='k', label=f'Abell 370 excess (R < 2′), N = {excess.sum():.0f}')
ax.step(db[:-1], frac(np.histogram(dmc[fc & hst_star], db)[0]), where='post', lw=1.5, label='COSMOS HST stars')
ax.step(db[:-1], frac(np.histogram(dmc[fc & ~hst_star], db)[0]), where='post', lw=1.5, label='COSMOS HST galaxies')
ax.axvline(0.145, c='0.5', ls='--', lw=1)                        # the SDSS type threshold (Chapter 1)
ax.set(xlabel='$r_{\\rm PSF} - r_{\\rm cModel}$', ylabel='fraction per 0.1 mag', title='all SDSS objects, 20 < r < 22.5')
ax.legend(fontsize=8)
plt.tight_layout()
# Fraction of clearly extended objects (Delta m >= 0.2). For the excess it is computed from the binned histogram
# (objects with Delta m outside [-0.4, 1.2] are left out); for COSMOS from all objects.
frac_ext = excess[db[:-1] >= 0.2].sum() / excess.sum()
print(f'fraction with Δm >= 0.2: Abell 370 excess {frac_ext:.2f}; COSMOS HST stars {np.mean(dmc[fc & hst_star] >= 0.2):.2f}; '
      f'COSMOS HST galaxies {np.mean(dmc[fc & ~hst_star] >= 0.2):.2f}')
# Test: the excess has more than twice the fraction of extended objects of real faint stars. Measured: 0.47 vs 0.11.
assert frac_ext > 2 * np.mean(dmc[fc & hst_star] >= 0.2)

**Figure 3.4.** Distribution of $\Delta m = r_{\rm PSF} - r_{\rm cModel}$ for SDSS objects of all types with
20 < $r_0$ < 22.5, in 0.1-mag bins from −0.4 to 1.2, each histogram normalised to unit sum: the Abell 370 excess
(black; objects within 2′ minus the 8′–19.8′ ring scaled by area, N = 145) compared with COSMOS SDSS objects of the
same magnitude that HST classifies as stars (blue) or galaxies (orange). The dashed grey line is the SDSS threshold
Δm = 0.145. Nearly half of the Abell 370 excess objects (0.47) have Δm ≥ 0.2, against 0.11 of faint stars (and 0.80
of faint galaxies): as a population they are extended objects, even though many of them individually pass as stars
(Δm < 0.145). The cluster members are also somewhat more compact than typical field galaxies of the same magnitude.

## Summary

- A compact, highly significant overdensity of "point sources" can be a galaxy cluster.
- Test the **excess** population, with the background subtracted: its colours (two-template likelihood for
  $f_\star$), its shapes compared with stars of the same magnitude, and — where available — a deeper survey.
- Build templates with the photometric errors of the target, including their correlation between colours.
- Colour templates are redshift dependent; the CMD (a turn-off, a giant branch, a horizontal branch at one
  distance) remains the strongest evidence for a stellar system.